# Hannah — listening test: names and languages (audio only, about 20 minutes)

This makes **sound only** (no video), so you can hear how Hannah would say visitors' names and how she
sounds in each language before we make hundreds of videos.

**Before you start (one time, free):**
1. Sign in or create a free account at **huggingface.co**
2. Open **huggingface.co/ai4bharat/IndicF5** and click **Agree and access repository**
3. Open **huggingface.co/settings/tokens** → **Create new token** → choose **Read** → **Create token** → **Copy**

**Run:**
1. **Runtime → Change runtime type → T4 GPU → Save** (if Colab says it cannot connect to a GPU, click **Connect without GPU**: this test still works)
2. **Runtime → Run all**
3. At step 2, paste the token into the box and press **Enter**
4. At the end, step 7 shows a player for every sample. Listen, then tell Claude your choices.


In [ ]:
# 1. GPU check (this test also works WITHOUT a GPU, just slower: about 40 minutes)
import subprocess, os
os.environ['MPLBACKEND'] = 'Agg'
try:
    gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
except FileNotFoundError:
    gpu = ''
print(gpu or 'No GPU: that is fine for this test. It will simply take longer. Carry on.')


In [ ]:
# 2. Your Hugging Face token (paste it in the box, press Enter)
from getpass import getpass
from huggingface_hub import login, hf_hub_download
login(token=getpass('Paste your Hugging Face token here and press Enter: ').strip())
try:
    hf_hub_download('ai4bharat/IndicF5', 'config.json')
    print('TOKEN OK: IndicF5 is unlocked')
except Exception as e:
    print('NOT UNLOCKED YET: open huggingface.co/ai4bharat/IndicF5, click "Agree and access repository", then run this cell again.')


In [ ]:
# 3. Install the two voice tools, each in its own clean Python 3.10 (8–15 minutes)
# Each part is installed on its own, so one that fails can't stop the others.
!apt-get -qq -y install espeak-ng ffmpeg cmake > /dev/null 2>&1
!pip install -q uv
# Kokoro: her current voice (and its Hindi, Spanish, French, Italian, Portuguese voices)
!uv venv -q --seed --python 3.10 /content/venv
!uv pip install -q --python /content/venv/bin/python torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install -q --python /content/venv/bin/python "kokoro>=0.9.4" "misaki[en]" soundfile "transformers<4.50"
!/content/venv/bin/python -m spacy download en_core_web_sm -q > /dev/null 2>&1
!/content/venv/bin/python -c "import kokoro, soundfile; print('KOKORO OK')"
# Chinese and Japanese add-ons (optional: if they fail, only those two samples are missing)
!uv pip install -q --python /content/venv/bin/python "misaki[zh]" && echo "CHINESE OK" || echo "CHINESE ADD-ON FAILED (that's fine)"
!uv pip install -q --python /content/venv/bin/python "misaki[ja]" && /content/venv/bin/python -m unidic download > /dev/null 2>&1 && echo "JAPANESE OK" || echo "JAPANESE ADD-ON FAILED (that's fine)"
# IndicF5: Indian languages, copying Hannah's own voice
!uv venv -q --seed --python 3.10 /content/venv_f5
!uv pip install -q --python /content/venv_f5/bin/python torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install -q --python /content/venv_f5/bin/python "git+https://github.com/ai4bharat/IndicF5.git" "transformers<4.50" "numpy<=1.26.4" "pyarrow==17.0.0" soundfile setuptools
!/content/venv_f5/bin/python -c "import f5_tts, torch, soundfile; print('INDICF5 OK, GPU:', torch.cuda.is_available())"


In [ ]:
# 4. What she will say in the test
import json, os
TEST = json.loads(r'''{"ref_en": "Hello and welcome to Nexora! I'm Hannah, your virtual receptionist, and I'll be your guide today.", "ref_hi": "नमस्ते, नेक्सोरा में आपका स्वागत है। मैं हन्ना हूँ, आपकी वर्चुअल रिसेप्शनिस्ट, और आज मैं आपकी गाइड हूँ।", "names": [{"en": "Divya", "hi": "दिव्या"}, {"en": "Rahul", "hi": "राहुल"}, {"en": "Sharan", "hi": "शरण"}, {"en": "Vinayak", "hi": "विनायक"}, {"en": "Gurpreet", "hi": "गुरप्रीत"}, {"en": "Lakshmi", "hi": "लक्ष्मी"}, {"en": "Emma"}, {"en": "Michael"}], "langs": [{"code": "en", "name": "English", "k": "a", "kokoro": ["af_bella"], "say": "Hello and welcome to Nexora! I'm Hannah, your virtual receptionist. May I know your name?"}, {"code": "hi", "name": "Hindi", "k": "h", "kokoro": ["af_bella", "hf_alpha"], "f5": ["en", "hi"], "say": "नमस्ते, नेक्सोरा में आपका स्वागत है! मैं हन्ना हूँ, आपकी वर्चुअल रिसेप्शनिस्ट। क्या मैं आपका नाम जान सकती हूँ?"}, {"code": "ta", "name": "Tamil", "f5": ["en", "hi"], "say": "வணக்கம், நெக்ஸோராவுக்கு உங்களை வரவேற்கிறேன்! நான் ஹன்னா, உங்கள் விர்ச்சுவல் வரவேற்பாளர். உங்கள் பெயரைத் தெரிந்துகொள்ளலாமா?"}, {"code": "te", "name": "Telugu", "f5": ["hi"], "say": "నమస్కారం, నెక్సోరాకు స్వాగతం! నేను హన్నా, మీ వర్చువల్ రిసెప్షనిస్ట్‌ని. మీ పేరు తెలుసుకోవచ్చా?"}, {"code": "kn", "name": "Kannada", "f5": ["hi"], "say": "ನಮಸ್ಕಾರ, ನೆಕ್ಸೋರಾಗೆ ಸ್ವಾಗತ! ನಾನು ಹನ್ನಾ, ನಿಮ್ಮ ವರ್ಚುವಲ್ ರಿಸೆಪ್ಷನಿಸ್ಟ್. ನಿಮ್ಮ ಹೆಸರು ತಿಳಿಯಬಹುದೇ?"}, {"code": "ml", "name": "Malayalam", "f5": ["hi"], "say": "നമസ്കാരം, നെക്സോറയിലേക്ക് സ്വാഗതം! ഞാൻ ഹന്ന, നിങ്ങളുടെ വെർച്വൽ റിസപ്ഷനിസ്റ്റ്. നിങ്ങളുടെ പേര് അറിയാമോ?"}, {"code": "mr", "name": "Marathi", "f5": ["hi"], "say": "नमस्कार, नेक्सोरामध्ये आपले स्वागत आहे! मी हन्ना, तुमची व्हर्च्युअल रिसेप्शनिस्ट. मी तुमचे नाव जाणून घेऊ शकते का?"}, {"code": "gu", "name": "Gujarati", "f5": ["hi"], "say": "નમસ્તે, નેક્સોરામાં આપનું સ્વાગત છે! હું હન્ના છું, તમારી વર્ચ્યુઅલ રિસેપ્શનિસ્ટ. શું હું તમારું નામ જાણી શકું?"}, {"code": "pa", "name": "Punjabi", "f5": ["hi"], "say": "ਸਤ ਸ੍ਰੀ ਅਕਾਲ, ਨੈਕਸੋਰਾ ਵਿੱਚ ਤੁਹਾਡਾ ਸੁਆਗਤ ਹੈ! ਮੈਂ ਹੰਨਾ ਹਾਂ, ਤੁਹਾਡੀ ਵਰਚੁਅਲ ਰਿਸੈਪਸ਼ਨਿਸਟ। ਕੀ ਮੈਂ ਤੁਹਾਡਾ ਨਾਮ ਜਾਣ ਸਕਦੀ ਹਾਂ?"}, {"code": "bn", "name": "Bengali", "f5": ["hi"], "say": "নমস্কার, নেক্সোরায় আপনাকে স্বাগত! আমি হান্না, আপনার ভার্চুয়াল রিসেপশনিস্ট। আপনার নামটা জানতে পারি?"}, {"code": "or", "name": "Odia", "f5": ["hi"], "say": "ନମସ୍କାର, ନେକ୍ସୋରାକୁ ଆପଣଙ୍କୁ ସ୍ୱାଗତ! ମୁଁ ହାନ୍ନା, ଆପଣଙ୍କ ଭର୍ଚୁଆଲ ରିସେପସନିଷ୍ଟ। ମୁଁ ଆପଣଙ୍କ ନାମ ଜାଣିପାରିବି କି?"}, {"code": "ur", "name": "Urdu", "f5": ["hi"], "say": "अस्सलाम अलैकुम, नेक्सोरा में ख़ुश आमदीद! मैं हन्ना हूँ, आपकी वर्चुअल रिसेप्शनिस्ट। क्या मैं आपका नाम जान सकती हूँ?"}, {"code": "es", "name": "Spanish", "k": "e", "kokoro": ["ef_dora", "af_bella"], "say": "¡Hola y bienvenido a Nexora! Soy Hannah, tu recepcionista virtual. ¿Puedo saber tu nombre?"}, {"code": "fr", "name": "French", "k": "f", "kokoro": ["ff_siwis", "af_bella"], "say": "Bonjour et bienvenue chez Nexora ! Je suis Hannah, votre réceptionniste virtuelle. Puis-je connaître votre prénom ?"}, {"code": "it", "name": "Italian", "k": "i", "kokoro": ["if_sara", "af_bella"], "say": "Ciao e benvenuto in Nexora! Sono Hannah, la tua receptionist virtuale. Posso sapere il tuo nome?"}, {"code": "pt", "name": "Portuguese", "k": "p", "kokoro": ["pf_dora", "af_bella"], "say": "Olá, seja bem-vindo à Nexora! Eu sou a Hannah, sua recepcionista virtual. Posso saber o seu nome?"}, {"code": "ja", "name": "Japanese", "k": "j", "kokoro": ["jf_alpha"], "say": "こんにちは、ネクソラへようこそ！私はハンナ、バーチャル受付です。お名前を伺ってもよろしいですか？"}, {"code": "zh", "name": "Chinese", "k": "z", "kokoro": ["zf_xiaobei"], "say": "您好，欢迎来到内克索拉！我是汉娜，您的虚拟接待员。请问您怎么称呼？"}]}''')
json.dump(TEST, open('/content/test.json', 'w'), ensure_ascii=False)
os.makedirs('/content/listen', exist_ok=True)
print(len(TEST['names']), 'names,', len(TEST['langs']), 'languages')


In [ ]:
%%writefile /content/kok.py
# 5a. Kokoro samples (Bella and the other Kokoro voices)
import json, numpy as np, soundfile as sf
from kokoro import KPipeline
T = json.load(open('/content/test.json'))
OUT = '/content/listen'
pipes = {}
pad = np.zeros(int(24000 * 0.25), dtype=np.float32)
def say(lang, voice, text, name):
    try:
        if lang not in pipes: pipes[lang] = KPipeline(lang_code=lang)
        a = np.concatenate([np.asarray(x) for _, _, x in pipes[lang](text, voice=voice, speed=0.95)])
        sf.write('%s/%s.wav' % (OUT, name), np.concatenate([pad, a, pad]), 24000)
        print('ok', name, flush=True)
    except Exception as e:
        print('FAILED', name, repr(e)[:200], flush=True)
# Bella's reference voice for IndicF5 (English, and Bella reading Hindi)
say('a', 'af_bella', T['ref_en'], 'ref_en')
say('h', 'af_bella', T['ref_hi'], 'ref_hi')
for n in T['names']:
    say('a', 'af_bella', n['en'] + '!', 'name_%s_A' % n['en'])
    if n.get('hi'): say('h', 'af_bella', n['hi'] + '!', 'name_%s_B' % n['en'])
for L in T['langs']:
    for v in L.get('kokoro', []):
        say(L['k'], v, L['say'], 'lang_%s_kokoro_%s' % (L['code'], v))
print('KOKORO SAMPLES DONE')


In [ ]:
!/content/venv/bin/python /content/kok.py 2>&1 | grep -E "^ok|FAILED|DONE|Error"


In [ ]:
%%writefile /content/f5.py
# 6a. IndicF5 samples: Indian languages in Hannah's (Bella's) voice
import json, numpy as np, soundfile as sf, torch
from transformers import AutoModel
T = json.load(open('/content/test.json'))
OUT = '/content/listen'
model = AutoModel.from_pretrained('ai4bharat/IndicF5', trust_remote_code=True)
try:
    if torch.cuda.is_available(): model = model.to('cuda')
except Exception as e:
    print('(running IndicF5 on CPU)', repr(e)[:100])
REF = {'en': ('/content/listen/ref_en.wav', T['ref_en']), 'hi': ('/content/listen/ref_hi.wav', T['ref_hi'])}
def say(text, ref, name):
    try:
        a = np.asarray(model(text, ref_audio_path=REF[ref][0], ref_text=REF[ref][1]))
        if a.dtype == np.int16: a = a.astype(np.float32) / 32768.0
        sf.write('%s/%s.wav' % (OUT, name), a.astype(np.float32), 24000)
        print('ok', name, flush=True)
    except Exception as e:
        print('FAILED', name, repr(e)[:300], flush=True)
for n in T['names']:
    if n.get('hi'): say(n['hi'] + '!', 'hi', 'name_%s_C' % n['en'])
for L in T['langs']:
    for ref in L.get('f5', []):
        say(L['say'], ref, 'lang_%s_indicf5_%sref' % (L['code'], ref))
print('INDICF5 SAMPLES DONE')


In [ ]:
!/content/venv_f5/bin/python /content/f5.py 2>&1 | grep -E "^ok|FAILED|DONE|Error|CPU"


In [ ]:
# 7. LISTEN: press play on each sample, then tell Claude your choices
import json, os
from IPython.display import display, HTML, Audio
T = json.load(open('/content/test.json'))
D = '/content/listen/'
def play(label, f):
    if os.path.exists(D + f):
        display(HTML('<div style="margin-top:8px">%s</div>' % label)); display(Audio(D + f))
    else:
        display(HTML('<div style="margin-top:8px;color:#c00">%s: not made (failed)</div>' % label))
display(HTML('<h2>Names: which of A, B, C says them best?</h2>'))
for n in T['names']:
    display(HTML('<h3>%s</h3>' % n['en']))
    play('A: Hannah, English way', 'name_%s_A.wav' % n['en'])
    if n.get('hi'):
        play('B: Hannah, Indian way', 'name_%s_B.wav' % n['en'])
        play('C: Hannah copied by IndicF5', 'name_%s_C.wav' % n['en'])
display(HTML('<h2>Languages: does she still sound like Hannah? Is it clear?</h2>'))
for L in T['langs']:
    display(HTML('<h3>%s</h3>' % L['name']))
    for v in L.get('kokoro', []):
        play('Kokoro ' + ('Bella (Hannah)' if v == 'af_bella' else 'native voice ' + v), 'lang_%s_kokoro_%s.wav' % (L['code'], v))
    for r in L.get('f5', []):
        play('IndicF5 copying Hannah (%s reference)' % ('English' if r == 'en' else 'Hindi'), 'lang_%s_indicf5_%sref.wav' % (L['code'], r))
!cd /content/listen && zip -q -r /content/hannah_voice_test.zip .
from google.colab import files
files.download('/content/hannah_voice_test.zip')
